In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

'from IPython.core.display import display, HTML\ndisplay(HTML("<style>.container { width:100% !important; }</style>"))'

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [167]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [ ]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("data/kg_train.csv",encoding='latin-1')


# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


### Let's divide the training and test set into two partitions

In [169]:
# Your code
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data['text'], data['label'], test_size=0.2, random_state=42)

In [170]:
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}\n")
print(f"Train label distribution:\n{y_train.value_counts(normalize=True)}")

Train size: 800, Test size: 200

Train label distribution:
label
0    0.54125
1    0.45875
Name: proportion, dtype: float64


## Data Preprocessing

In [171]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [172]:
# Your code
import re
def text_processing_1(text):
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'<script[^>]*>.*?</script>', '', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'<style[^>]*>.*?</style>', '', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)
    text = re.sub(r'<.*?>', '', text, flags=re.DOTALL)
    return text



X_train = X_train.apply(text_processing_1)
X_test = X_test.apply(text_processing_1)

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [173]:
# Your code
def text_processing_2(text):
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\b\w\b', ' ', text)
    text = re.sub(r'\A\w\s', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'^b\s|^B\s', '', text)
    text = str(text).lower()
    return text



X_train = X_train.apply(text_processing_2)
X_test = X_test.apply(text_processing_2)

X_test
X_train

29     regards mr nelson smith kindly reply me on my ...
535    have not been able to reach oscar this am we a...
695    huma abedin checking with pat on the will work...
557       can have it announced here on monday can today
836    bank of africaagence san pedro bp san pedro co...
                             ...                        
106    adama ibrahim_________________________________...
270                what does that mean for our schedules
860    dear friend my compliment to you guess this le...
435    dear president fdirector my name is mr micheal...
102    let me know if today or tomorrow works for you...
Name: text, Length: 800, dtype: object

## Now let's work on removing stopwords
Remove the stopwords.

In [174]:
# Your code
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words('english'))

def text_processing_3(text):
    tokens = word_tokenize(text)
    tokens = [str(t) for t in tokens if t not in stop_words]
    return tokens

X_train = X_train.apply(text_processing_3)
X_test = X_test.apply(text_processing_3)

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [175]:
# Your code
from nltk.corpus import wordnet
import nltk

def get_wordnet_pos_from_tag(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN
    

from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

def text_processing_4(tokens):
    if not tokens:
        return ''
    return ' '.join([lemmatizer.lemmatize(token) for token in tokens])


X_train = X_train.apply(text_processing_4)
X_test = X_test.apply(text_processing_4)


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [176]:
# Your code
train_df = pd.DataFrame({'text': X_train, 'label': y_train})

ham_text = ' '.join(train_df[train_df['label'] == 0]['text'])
spam_text = ' '.join(train_df[train_df['label'] == 1]['text'])

from collections import Counter

def get_top_words(text, n=10):
    words = text.split()
    word_counts = Counter(words)
    return word_counts.most_common(n)

print("Top 10 words in HAM messages:")
for word, count in get_top_words(ham_text, 10):
    print(f"  {word}: {count}")

print("\nTop 10 words in SPAM messages:")
for word, count in get_top_words(spam_text, 10):
    print(f"  {word}: {count}")

Top 10 words in HAM messages:
  state: 117
  pm: 97
  would: 94
  president: 89
  mr: 89
  time: 81
  percent: 80
  obama: 77
  call: 74
  secretary: 74

Top 10 words in SPAM messages:
  money: 842
  account: 740
  bank: 645
  fund: 625
  u: 594
  transaction: 466
  business: 422
  mr: 421
  country: 419
  million: 369


## Extra features

In [177]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])


data_train = pd.DataFrame({'preprocessed_text': X_train, 'label': y_train})
data_val = pd.DataFrame({'preprocessed_text': X_test, 'label': y_test})


data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,preprocessed_text,label,money_mark,suspicious_words,text_len
29,regard mr nelson smith kindly reply private em...,1,0,0,79
535,able reach oscar supposed send pdb receive,0,0,0,42
695,huma abedin checking pat work jack jake rest a...,0,0,0,76
557,announced monday today,0,0,0,22
836,bank africaagence san pedro bp san pedro cote ...,1,1,1,1082


## How would work the Bag of Words with Count Vectorizer concept?

In [178]:
# Your code
from sklearn.feature_extraction.text import CountVectorizer
from scipy.sparse import hstack

# Create bag-of-words from the preprocessed text column
bow_vectorizer = CountVectorizer(lowercase=True, max_features=1000)

X_train_bow = bow_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_bow   = bow_vectorizer.transform(data_val['preprocessed_text'])

# Extra features as arrays
X_train_extra = data_train[['money_mark', 'suspicious_words', 'text_len']].values
X_val_extra   = data_val[['money_mark', 'suspicious_words', 'text_len']].values

# Combine features horizontally
X_train_combined = hstack([X_train_bow, X_train_extra])
X_val_combined   = hstack([X_val_bow, X_val_extra])

print("Bag-of-Words + extra features")
print(f"Training shape: {X_train_combined.shape}")
print(f"Validation shape: {X_val_combined.shape}")

Bag-of-Words + extra features
Training shape: (800, 1003)
Validation shape: (200, 1003)


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [179]:
# Your code
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# Directly use TfidfVectorizer (count + tf-idf in one step)
tfidf_vectorizer = TfidfVectorizer(lowercase=True, max_features=1000)

X_train_tfidf = tfidf_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_tfidf   = tfidf_vectorizer.transform(data_val['preprocessed_text'])

# Combine with extra features (same extra arrays as above)
X_train_tfidf_combined = hstack([X_train_tfidf, X_train_extra])
X_val_tfidf_combined   = hstack([X_val_tfidf, X_val_extra])

print("TF-IDF + extra features")
print(f"Training shape: {X_train_tfidf_combined.shape}")
print(f"Validation shape: {X_val_tfidf_combined.shape}")

TF-IDF + extra features
Training shape: (800, 1003)
Validation shape: (200, 1003)


## And the Train a Classifier?

In [180]:
# Your code
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Choose which feature matrix to use:
# Option 1: Bag-of-Words + extra features
X_train = X_train_combined
X_val   = X_val_combined

# Option 2: TF-IDF + extra features (uncomment the next two lines)
# X_train = X_train_tfidf_combined
# X_val   = X_val_tfidf_combined

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_val)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

Accuracy: 0.9600
              precision    recall  f1-score   support

         Ham       0.95      0.99      0.97       125
        Spam       0.99      0.91      0.94        75

    accuracy                           0.96       200
   macro avg       0.97      0.95      0.96       200
weighted avg       0.96      0.96      0.96       200



### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [181]:
# Your code